In [ ]:
!pip install pandas numpy scikit-learn catboost lightgbm xgboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 7.6 MB/s eta 0:00:00


In [ ]:
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import ExtraTreesRegressor, GradientBoostingRegressor
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.metrics import root_mean_squared_error
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Зададим повторимость результата в переменную
RANDOM_STATE = 42
N_FOLDS = 5

N_ITER = {'et': 1000, 'cb': 1200, 'xgb': 800, 'gbr': 600, 'lgbm': 800}

BEST_SEEDS = {
    'ic50': 32,
    'cc50': 41,
    'si': 1
}

# Загрузка данных
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
sample_submission = pd.read_csv('sample_submission.csv')


target_cols = ['IC50, mM', 'CC50, mM', 'SI']
feature_cols = [col for col in train.columns if col not in target_cols + ['ID']]

X_train_full = train[feature_cols].values
X_test = test[feature_cols].values
y_train = train[target_cols]

y_ic50 = y_train['IC50, mM'].values
y_cc50 = y_train['CC50, mM'].values
y_si = y_train['SI'].values

# Предобработка
imp = SimpleImputer(strategy='median') # пропуски заполняем медианой
X_train_imp = imp.fit_transform(X_train_full)
X_test_imp = imp.transform(X_test) # заполнение пропусков в тестовых данных медианой по тренировочным

# Отбор признаков для каждой целевой переменной, 100 признаков дают лучшие метрики
selector_ic50 = SelectFromModel(
    ExtraTreesRegressor(n_estimators=600, max_features=0.7, min_samples_leaf=2,
                        bootstrap=False, random_state=RANDOM_STATE, n_jobs=-1),
    threshold=-np.inf, max_features=100
)
selector_cc50 = SelectFromModel(
    ExtraTreesRegressor(n_estimators=600, max_features=0.7, min_samples_leaf=2,
                        bootstrap=False, random_state=RANDOM_STATE+1, n_jobs=-1),
    threshold=-np.inf, max_features=100
)
selector_si = SelectFromModel(
    ExtraTreesRegressor(n_estimators=600, max_features=0.7, min_samples_leaf=2,
                        bootstrap=False, random_state=RANDOM_STATE+2, n_jobs=-1),
    threshold=-np.inf, max_features=100
)

X_train_ic50 = selector_ic50.fit_transform(X_train_imp, y_ic50)
X_test_ic50 = selector_ic50.transform(X_test_imp)

X_train_cc50 = selector_cc50.fit_transform(X_train_imp, y_cc50)
X_test_cc50 = selector_cc50.transform(X_test_imp)

X_train_si = selector_si.fit_transform(X_train_imp, y_si)
X_test_si = selector_si.transform(X_test_imp)

# Ансамбль моделей
def get_models(seed):
    return [
        ('et', ExtraTreesRegressor(n_estimators=N_ITER['et'], max_features=0.7,
                                   min_samples_leaf=3, bootstrap=False,
                                   random_state=seed, n_jobs=-1)),
        ('cb', CatBoostRegressor(iterations=N_ITER['cb'], learning_rate=0.03,
                                 depth=4, l2_leaf_reg=10, random_seed=seed, verbose=0)),
        ('xgb', XGBRegressor(n_estimators=N_ITER['xgb'], learning_rate=0.03,
                             max_depth=4, subsample=0.8, colsample_bytree=0.8,
                             reg_lambda=1.0, random_state=seed, n_jobs=-1)),
        ('gbr', GradientBoostingRegressor(n_estimators=N_ITER['gbr'], learning_rate=0.05,
                                          max_depth=4, subsample=0.8, min_samples_leaf=2,
                                          random_state=seed)),
        ('lgbm', LGBMRegressor(n_estimators=N_ITER['lgbm'], learning_rate=0.03,
                               num_leaves=31, subsample=0.8, colsample_bytree=0.8,
                               reg_lambda=1.0, random_state=seed, verbose=-1, n_jobs=-1))
    ]

# Обучение, по фиксированным сидам для каждой целевой переменной
def train_stack(X, y, X_test, seed, target_name):
    models = get_models(seed)

    if target_name == 'si':
        y_binned = pd.qcut(y, q=5, labels=False, duplicates='drop')
        kf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
        fold_splits = list(kf.split(X, y_binned))
    else:
        kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
        fold_splits = list(kf.split(X))

    oof_preds = np.zeros((X.shape[0], len(models)))
    test_preds = np.zeros((X_test.shape[0], len(models)))

    for i, (name, model) in enumerate(models):
        test_folds = np.zeros((X_test.shape[0], N_FOLDS))

        for fold, (tr, val) in enumerate(fold_splits):
            clone = model.__class__(**model.get_params())
            clone.fit(X[tr], y[tr])
            oof_preds[val, i] = clone.predict(X[val])
            test_folds[:, fold] = clone.predict(X_test)

        test_preds[:, i] = np.mean(test_folds, axis=1)

    if target_name == 'si':
        meta = RidgeCV(alphas=[0.1, 1.0, 10.0], cv=5).fit(
            np.log1p(np.maximum(oof_preds, 0)),
            np.log1p(y)
        )
        oof = np.expm1(meta.predict(np.log1p(np.maximum(oof_preds, 0))))
        test = np.expm1(meta.predict(np.log1p(np.maximum(test_preds, 0))))
    else:
        meta = RidgeCV(alphas=[0.1, 1.0, 10.0], cv=5).fit(oof_preds, y)
        oof = meta.predict(oof_preds)
        test = meta.predict(test_preds)

    return oof, test, oof_preds

# Обучаем, с получением метрик на тренировочных данных
oof_ic50, pred_ic50, oof_preds_ic50 = train_stack(X_train_ic50, y_ic50, X_test_ic50, BEST_SEEDS['ic50'], 'ic50')
oof_cc50, pred_cc50, oof_preds_cc50 = train_stack(X_train_cc50, y_cc50, X_test_cc50, BEST_SEEDS['cc50'], 'cc50')
oof_si, pred_si, oof_preds_si = train_stack(X_train_si, y_si, X_test_si, BEST_SEEDS['si'], 'si')

# Легкая коррекция SI
si_formula_test = pred_cc50 / (pred_ic50 + 1e-9)
pred_si = 0.95 * pred_si + 0.05 * si_formula_test

# Клиппинг, смягчение экстремальных значений
pred_ic50 = np.clip(pred_ic50, 1e-9, np.percentile(y_ic50, 99.9))
pred_cc50 = np.clip(pred_cc50, 1e-9, np.percentile(y_cc50, 99.9))
pred_si = np.clip(pred_si, 1e-9, np.percentile(y_si, 99.9))

In [ ]:
# Вычисляем общий RMSE на тренировочных данных
rmse_ic50 = root_mean_squared_error(y_ic50, oof_ic50)
rmse_cc50 = root_mean_squared_error(y_cc50, oof_cc50)
rmse_si = root_mean_squared_error(y_si, oof_si)
avg_rmse = (rmse_ic50 + rmse_cc50 + rmse_si) / 3

print(f'OOF RMSE: IC50={rmse_ic50:.4f}, CC50={rmse_cc50:.4f}, SI={rmse_si:.4f}, Avg={avg_rmse:.4f}')

# Вывод OOF RMSE для каждой модели
print("\n--- RMSE отдельных моделей на тренировочных данных (OOF) ---")
model_names = [name for name, _ in get_models(BEST_SEEDS['ic50'])]  # имена моделей

print("\nIC50:")
for i, name in enumerate(model_names):
    rmse = root_mean_squared_error(y_ic50, oof_preds_ic50[:, i])
    print(f"  {name}: {rmse:.4f}")

print("\nCC50:")
for i, name in enumerate(model_names):
    rmse = root_mean_squared_error(y_cc50, oof_preds_cc50[:, i])
    print(f"  {name}: {rmse:.4f}")

print("\nSI:")
for i, name in enumerate(model_names):
    rmse = root_mean_squared_error(y_si, oof_preds_si[:, i])
    print(f"  {name}: {rmse:.4f}")

OOF RMSE: IC50=309.8008, CC50=446.3594, SI=788.1076, Avg=514.7560

--- RMSE отдельных моделей на тренировочных данных (OOF) ---

IC50:
  et: 313.0446
  cb: 327.5112
  xgb: 330.5780
  gbr: 356.1595
  lgbm: 337.0042

CC50:
  et: 450.0528
  cb: 456.7136
  xgb: 465.7532
  gbr: 474.8070
  lgbm: 476.3628

SI:
  et: 791.4148
  cb: 915.7337
  xgb: 924.0616
  gbr: 1018.2874
  lgbm: 837.3421


In [ ]:
# Сохраняем результат
sample = sample_submission.copy()
sample['IC50'] = pred_ic50
sample['CC50'] = pred_cc50
sample['SI'] = pred_si
sample.to_csv('best_271_81.csv', index=False)